# Introducción al Machine Learning

El **Machine Learning** (aprendizaje automático) es una rama de la Inteligencia Artificial que permite que las máquinas **aprendan patrones a partir de los datos**, sin necesidad de ser programadas explícitamente para cada tarea. En lugar de seguir reglas fijas, un modelo aprende de ejemplos y luego **generaliza** para hacer predicciones o tomar decisiones sobre datos nuevos.

## ¿Para qué sirve?

- **Clasificación**: predecir una categoría o clase (por ejemplo, si un empleado abandonará o no la empresa).
- **Regresión**: predecir un valor numérico (por ejemplo, el precio de una vivienda).
- **Agrupamiento (clustering)**: descubrir grupos en los datos sin etiquetas previas.
- **Detección de anomalías**, **sistemas de recomendación**, **visión por computadora**, etc.

## Tipos de entrenamiento

| Tipo | Descripción | Ejemplos |
| --- | --- | --- |
| **Supervisado** | Aprende de datos etiquetados (se conoce la respuesta correcta). | Clasificación y regresión |
| **No supervisado** | Descubre patrones en datos sin etiquetar. | Clustering, reducción de dimensionalidad |
| **Por refuerzo** | Aprende por ensayo y error maximizando una recompensa. | Juegos, robótica |

## Flujo típico de un proyecto de Machine Learning

1. Cargar y entender los datos.
2. Explorar y limpiar los datos (EDA).
3. Preparar las variables: codificar categóricas y escalar numéricas.
4. Dividir los datos en **entrenamiento** y **prueba**.
5. Entrenar el modelo.
6. Evaluar el modelo y, si tiene buen desempeño, **publicarlo** para su uso (por ejemplo, en una Web API).


## Ejercicio práctico: Predecir si un emplado abanonará su trabajo

En este ejercicio utilizaremos el modelo de clasificación k-NN que clasifica observaciones basándose en los k vecinos más cercanos.

Es necesario instalar libreria Sklearn:
```python
python -m pip install scikit-learn
```

In [ ]:
!python -m pip install scikit-learn

## Paso 1: Cargar y preparar datos

En este paso **cargamos el dataset** con `pandas` y lo **preparamos**:

- `pd.read_csv(...)`: lee el archivo CSV (usa `sep=";"` porque el archivo separa las columnas con punto y coma).
- `data.columns` y `data.dtypes`: muestran el nombre y el tipo de cada columna.
- `isna().sum()`: cuenta los valores nulos por columna.
- `drop(...)`: elimina columnas que no aportan al análisis (por ejemplo, `anos_en_puesto` y `conciliacion`, que tienen muchos nulos).

In [ ]:
import pandas as pd

# Cargar los datos
data = pd.read_csv("./data/abandono_empleados.csv", sep=";")
# Mostrando las columnas
print(data.columns)
print(data.dtypes)
data.head()

In [ ]:
# Revisando datos nulos
data.isna().sum().sort_values(ascending=False)

In [ ]:
# Limpiando datos
data = data.drop(columns=["anos_en_puesto", "conciliacion"])
data.head()

## Paso 2: Análisis Exploratorio (EDA)

Aquí **exploramos los datos** para entenderlos antes de modelar:

- Se generan **gráficos de barras** de las variables categóricas para ver su distribución.
- Se eliminan columnas **constantes**, que no aportan información (`mayor_edad`, `empleados` y `horas_quincena`).
- Se calculan **estadísticos descriptivos** de las variables numéricas (`describe` + mediana).
- Se analiza la **variable objetivo** (`abandono`) y su relación con educación, estado civil, horas extras, puesto, viajes y salario.

In [ ]:
# Explorar las los atributos relevantes
def graficos_eda_categoricos(cat):
    #Calculamos el número de filas que necesitamos
    from math import ceil
    filas = ceil(cat.shape[1] / 2)

    #Definimos el gráfico
    f, ax = plt.subplots(nrows = filas, ncols = 2, figsize = (16, filas * 6))

    #Aplanamos para iterar por el gráfico como si fuera de 1 dimensión en lugar de 2
    ax = ax.flat

    #Creamos el bucle que va añadiendo gráficos
    for cada, variable in enumerate(cat):
        cat[variable].value_counts().plot.barh(ax = ax[cada])
        ax[cada].set_title(variable, fontsize = 12, fontweight = "bold")
        ax[cada].tick_params(labelsize = 12)

In [ ]:
import matplotlib.pyplot as plt

graficos_eda_categoricos(data.select_dtypes('O'))

In [ ]:
# Conclusiones de la exploración de datos
# - Eliminar columna "mayor_edad" porque es un valor consante
data = data.drop("mayor_edad", axis=1)
data.head()

In [ ]:
# Mostrar las estadisticas de los valores númericos
def estadisticos_cont(num):
    #Calculamos describe
    estadisticos = num.describe().T
    #Añadimos la mediana
    estadisticos['median'] = num.median()
    #Reordenamos para que la mediana esté al lado de la media
    estadisticos = estadisticos.iloc[:,[0,1,8,2,3,4,5,6,7]]
    #Lo devolvemos
    return(estadisticos)

estadisticos_cont(data.select_dtypes('number'))

In [ ]:
# Conclusiones del análisis estadístico de los datos
# - Eliminar columnas "empleados" y "horas_quincena"  porque tienen valores consantes
data = data.drop(columns=["empleados", "horas_quincena"])
data.head()

In [ ]:
# Ver el comportamiento de la variable dependiente (abando)
data.abandono.value_counts(normalize = True) * 100

In [ ]:
# Análisis por Educación
df_edu = data[data["abandono"]=='Yes'].groupby("educacion").abandono.count().sort_values(ascending=False)
df_edu.plot.bar()

In [ ]:
# Análisis por estado civil
df_ec = data[data["abandono"]=='Yes'].groupby("estado_civil").abandono.count().sort_values(ascending=False)
df_ec.plot.bar()

In [ ]:
# Analisis por horas extras
df_hr = data[data["abandono"]=='Yes'].groupby("horas_extra").abandono.count().sort_values(ascending=False)
df_hr.plot.bar()

In [ ]:
# Analisis por puesto
df_p = data[data["abandono"]=='Yes'].groupby("puesto").abandono.count().sort_values(ascending=False)
df_p.plot.bar()

In [ ]:
# Analisis por viaje
df_v = data[data["abandono"]=='Yes'].groupby("viajes").abandono.count().sort_values(ascending=False)
df_v.plot.bar()

In [ ]:
# Análisis de salario promedio por abandono
df_s = data.groupby('abandono').salario_mes.mean()
df_s.plot.bar()

In [ ]:
# Mostrando las columnas de dataset limpio
data.dtypes

## Paso 3: Codificar y Segmentar datos

En este paso **preparamos los datos para el modelo**, que solo trabaja con números:

- `LabelEncoder`: convierte las **categorías en números** (por ejemplo, `Yes`/`No` → `1`/`0`).
- Se define `X` (variables de entrada, sin la columna `abandono`) y `y` (la variable a predecir: `abandono`).
- `train_test_split`: separa los datos en **entrenamiento** (80 %) y **prueba** (20 %).
- `StandardScaler`: **escala** las variables para que todas tengan la misma magnitud; es importante para k-NN, que mide distancias.

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Clasificación de variable cualitativas (categorias)
categorical_columns = ["abandono", "viajes", "departamento", "educacion", "carrera", "satisfaccion_entorno", 
                       "sexo", "implicacion", "puesto", "satisfaccion_trabajo", "estado_civil", "horas_extra", "evaluacion", 
                       "satisfaccion_companeros"]
labels_econders = {}
for col in categorical_columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    labels_econders[col] = le

labels_econders

In [ ]:
from sklearn.model_selection import train_test_split

# Definir variable independiente
X = data.drop("abandono", axis=1)
# Definir variable dependiente
y = data["abandono"]

# Divir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalando las caracteristicas
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)


## Paso 4: Entrenar el modelo

Aquí **entrenamos el modelo**:

- `KNeighborsClassifier(n_neighbors=5)`: crea el clasificador k-NN que decidirá según los **5 vecinos más cercanos**.
- `knn.fit(X_train, y_train)`: el modelo **aprende** los patrones del conjunto de entrenamiento.

In [ ]:
# Entrenamiento del modelo
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

## Paso 5: Evaluar el modelo

En este paso **evaluamos el modelo** con datos que no ha visto durante el entrenamiento:

- `knn.predict(X_test)`: genera las **predicciones** sobre el conjunto de prueba.
- `classification_report`: compara las predicciones con las etiquetas reales y muestra **precisión, recall y f1-score**.
- Después, en la siguiente celda, haremos una **predicción con un dato real**.

In [ ]:
# Evaluando el modelo
# Predicciones
y_pred = knn.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# Predicción con un dato real: un nuevo empleado
# Preparamos un empleado de ejemplo con las mismas columnas que usó el modelo
empleado_nuevo = pd.DataFrame([{
    'edad': 34,
    'viajes': 'Travel_Rarely',
    'departamento': 'Sales',
    'distancia_casa': 4,
    'educacion': 'Universitaria',
    'carrera': 'Marketing',
    'id': 9999,
    'satisfaccion_entorno': 'Alta',
    'sexo': 2.0,
    'implicacion': 'Alta',
    'nivel_laboral': 2,
    'puesto': 'Sales Executive',
    'satisfaccion_trabajo': 'Baja',
    'estado_civil': 'Single',
    'salario_mes': 2600,
    'num_empresas_anteriores': 3,
    'horas_extra': 'Yes',
    'incremento_salario_porc': 12,
    'evaluacion': 'Alta',
    'satisfaccion_companeros': 'Alta',
    'nivel_acciones': 1,
    'anos_experiencia': 6,
    'num_formaciones_ult_ano': 2,
    'anos_compania': 2,
    'anos_desde_ult_promocion': 1,
    'anos_con_manager_actual': 1,
}])

# Aseguramos el mismo orden de columnas que usó el modelo
empleado_nuevo = empleado_nuevo[X.columns]

# Codificamos las variables categóricas con los codificadores guardados
for col in categorical_columns:
    if col != "abandono":
        empleado_nuevo[col] = labels_econders[col].transform(empleado_nuevo[col])

# Escalamos con el mismo escalador del entrenamiento
empleado_nuevo_escalado = scaler.transform(empleado_nuevo)

# Predecimos si el nuevo empleado abandonará la empresa
prediccion = knn.predict(empleado_nuevo_escalado)[0]
resultado = labels_econders["abandono"].inverse_transform([prediccion])[0]

print("Predicción de abandono para el nuevo empleado:", resultado)

## Paso 6: Publicar el modelo

Para que el modelo pueda utilizarse desde otras aplicaciones, se **serializa** (se guarda en disco) junto con todas sus transformaciones (codificadores y escalador) y luego se expone mediante una **Web API**. De esta forma, cualquier sistema puede enviar los datos de un empleado y recibir la predicción sin necesidad de volver a entrenar el modelo.

In [ ]:
import joblib

# Guardamos el modelo y todo lo necesario para predecir
artefactos = {
    "modelo": knn,
    "escalador": scaler,
    "codificadores": labels_econders,
    "columnas": list(X.columns),
}

joblib.dump(artefactos, "modelo_abandono.joblib")
print("Modelo y transformaciones guardados en 'modelo_abandono.joblib'")

# Verificamos que se puede volver a cargar
cargado = joblib.load("modelo_abandono.joblib")
print("Modelo cargado correctamente:", type(cargado["modelo"]).__name__)
print("Columnas esperadas por el modelo:", cargado["columnas"])

### Web API con FastAPI

Creamos una API que recibe los datos de un empleado y devuelve la predicción de abandono. Escribimos el código en un archivo para poder ejecutarlo como un servicio.

In [ ]:
%%writefile api_abandono.py
# Web API para predecir el abandono de empleados
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

# Cargamos el modelo y sus transformaciones
artefactos = joblib.load("modelo_abandono.joblib")
modelo = artefactos["modelo"]
escalador = artefactos["escalador"]
codificadores = artefactos["codificadores"]
columnas = artefactos["columnas"]

app = FastAPI(title="API de predicción de abandono de empleados")


class Empleado(BaseModel):
    edad: int
    viajes: str
    departamento: str
    distancia_casa: int
    educacion: str
    carrera: str
    id: int
    satisfaccion_entorno: str
    sexo: float
    implicacion: str
    nivel_laboral: int
    puesto: str
    satisfaccion_trabajo: str
    estado_civil: str
    salario_mes: int
    num_empresas_anteriores: int
    horas_extra: str
    incremento_salario_porc: int
    evaluacion: str
    satisfaccion_companeros: str
    nivel_acciones: int
    anos_experiencia: int
    num_formaciones_ult_ano: int
    anos_compania: int
    anos_desde_ult_promocion: int
    anos_con_manager_actual: int


@app.get("/")
def raiz():
    return {"mensaje": "API de predicción de abandono de empleados"}


@app.post("/predecir")
def predecir(empleado: Empleado):
    # Convertimos a DataFrame respetando el orden de columnas del modelo
    fila = pd.DataFrame([empleado.model_dump()])[columnas]

    # Codificamos las variables categóricas
    for col in codificadores:
        if col != "abandono":
            fila[col] = codificadores[col].transform(fila[col])

    # Escalamos y predecimos
    fila_escalada = escalador.transform(fila)
    prediccion = modelo.predict(fila_escalada)[0]
    resultado = codificadores["abandono"].inverse_transform([prediccion])[0]

    return {"prediccion_abandono": resultado}

Para ejecutar la API desde una terminal:

```bash
uvicorn api_abandono:app --reload
```

Luego se puede probar con una petición POST:

```python
import requests

empleado = {
    "edad": 34, "viajes": "Travel_Rarely", "departamento": "Sales",
    "distancia_casa": 4, "educacion": "Universitaria", "carrera": "Marketing",
    "id": 9999, "satisfaccion_entorno": "Alta", "sexo": 2.0,
    "implicacion": "Alta", "nivel_laboral": 2, "puesto": "Sales Executive",
    "satisfaccion_trabajo": "Baja", "estado_civil": "Single",
    "salario_mes": 2600, "num_empresas_anteriores": 3, "horas_extra": "Yes",
    "incremento_salario_porc": 12, "evaluacion": "Alta",
    "satisfaccion_companeros": "Alta", "nivel_acciones": 1,
    "anos_experiencia": 6, "num_formaciones_ult_ano": 2,
    "anos_compania": 2, "anos_desde_ult_promocion": 1,
    "anos_con_manager_actual": 1,
}

respuesta = requests.post("http://127.0.0.1:8000/predecir", json=empleado)
print(respuesta.json())
```